In [1]:
## Daniel clone, records unknown questions and gets user interests (?)
# imports

from dotenv import load_dotenv
from anthropic import Anthropic
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

## Notification feature (skip)

In [ ]:
# For pushover - Secondary

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

## Text inputs

In [3]:
# name: str
name = "Daniel"

In [4]:
# Input artifacts (str)
reader = PdfReader("4_lab4_daniel/profile.pdf")
document = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        document += text

with open("4_lab4_daniel/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


## Tools

In [5]:
# Description of requirements, must-to-have inputs, using context engineering
DEFAULT_NAME = "Name not provided"
DEFAULT_NOTES = "not provided"
def record_user_details(email: str, name: str=DEFAULT_NAME, notes: str=DEFAULT_NOTES) -> str:
    if not email:
        raise ValueError("Email required to record user details")
    with open("4_lab4_daniel/user_details.txt", "a", encoding="utf-8") as f:
        user = {
            "email": email,
            "name": name,
            "notes": notes,
        }
        f.write(f"{user}\n")
    return "User details stored"

record_user_details_tool = {
    "name": "record_user_details",
    "description": "Record the details of the user once he/she provides the email and sufficient information to record",
    "input_schema": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}
    

In [6]:
# Given context engineering, it detects -> extracts -> append -> unkwnown question
def record_unknown_question(question: str) -> str:
    if not question:
        raise ValueError("Question required to record unknown question")
    with open("4_lab4_daniel/unknown_questions.txt", "a", encoding="utf-8") as f:
        unknow_question = question
        f.write(f"{unknow_question}\n")
    return "Unknown question recorded"

record_unknown_question_tool = {
    "name": "record_unknown_question",
    "description": "Record any question that couldn't be answered as you didn't know the answer despite the context you have",
    "input_schema": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
    }
}

In [7]:
tools = [record_user_details_tool, record_unknown_question_tool]

In [8]:
system_prompt = f"You are acting as {name}. You are a senior AI Agentic Engineer. You will provide support and insghts for AI, Cyber security."

system_prompt += f"\n\n## Document:\n{document}\n\n"
system_prompt += f"\n\n## Summary:\n{summary}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}. Be short, concise, and provide actionable insights."
system_prompt += f"\n\n If you don't know the answer, say 'I don't know' instead of making up an answer."


In [9]:
# Dispatcher  — canonical, works for 1 or N tools
def run_tool(name: str, tool_input: dict) -> str:
    if name == "record_user_details":
        return record_user_details(email=tool_input["email"],
                                   name=tool_input.get("name", DEFAULT_NAME),
                                   notes=tool_input.get("notes", DEFAULT_NOTES),
                                   ) # type: ignore
    if name == "record_unknown_question":
        return record_unknown_question(question=tool_input["question"])
    raise ValueError(f"Unknown tool: {name}")

In [ ]:
# Chat function (Gradio) with tool support
def chat(message, history) -> str | None:
    # Chat logging (history)
    history = [{"role": h["role"], "content": h["content"]} for h in history] # Keep only role, content. Drop Gradio's extra keys
    messages = history + [{"role": "user", "content": message}] # Append the user turn

    while True: # Continuous loop
        response = claude.messages.create(
            model=model_name,
            messages=messages,
            timeout=59,
            max_tokens=1024,
            system=system_prompt,
            tools=tools, # List of tools
        )
        if response.stop_reason != "tool_use": # Checks whethet Claude is done (not requesting a tool)
            return next(b.text for b in response.content if b.type == "text") # FINISH LOOP: Return the text of the first block

        messages.append({"role": "assistant", "content": response.content}) # Record Claude's turn (text + tool_use blocks) in the conversation

        tool_results = [] # Collect one tool_result per tool_use block below
        for block in response.content:  # Scan every block in Claude's reply (text and tool_use)
            print(block)
            if block.type == 'tool_use':
                result = run_tool(block.name, block.input) # RUN TOOL THROUGH DISPATCHER: block.name, block.input are set by Claude
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
        messages.append({"role": "user", "content": tool_results})  # Send all tool results back as one user message

In [11]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## And now for deployment

This code is in `app.py`

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the "me" directory - your LinkedIn profile and summary.txt - so that it talks about you! Also change `self.name = "Ed Donner"` in `app.py`..  

Also check that there's no README file within the 1_foundations directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

And one more thing: this is optional, but you might want to delete the entire folder "community_contributions" within 1_foundations. You can always pull it from github again in the future. But if you don't, then this entire folder gets uploaded to HuggingFace even though we don't need it, and it's become quite large.

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Terminal, run: `uv tool install 'huggingface_hub[cli]'` to install the HuggingFace tool, then `hf auth login --token YOUR_TOKEN_HERE`, like `hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future
5. From the 1_foundations folder, enter: `uv run gradio deploy` 
6. Follow its instructions: name it "career_conversation", specify app.py, choose cpu-basic as the hardware, say Yes to needing to supply secrets, provide your openai api key, your pushover user and token, and say "no" to github actions.  

Thank you Robert, James, Martins, Andras and Priya for these tips.  
Please read the next 2 sections - how to change your Secrets, and how to redeploy your Space (you may need to delete the README.md that gets created in this 1_foundations directory).

#### More about these secrets:

If you're confused by what's going on with these secrets: it just wants you to enter the key name and value for each of your secrets -- so you would enter:  
`OPENAI_API_KEY`  
Followed by:  
`sk-proj-...`  

And if you don't want to set secrets this way, or something goes wrong with it, it's no problem - you can change your secrets later:  
1. Log in to HuggingFace website  
2. Go to your profile screen via the Avatar menu on the top right  
3. Select the Space you deployed  
4. Click on the Settings wheel on the top right  
5. You can scroll down to change your secrets (Variables and Secrets section), delete the space, etc.

#### And now you should be deployed!

If you want to completely replace everything and start again with your keys, you may need to delete the README.md that got created in this 1_foundations folder.

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

To delete your Space in the future:  
1. Log in to HuggingFace
2. From the Avatar menu, select your profile
3. Click on the Space itself and select the settings wheel on the top right
4. Scroll to the Delete section at the bottom
5. ALSO: delete the README file that Gradio may have created inside this 1_foundations folder (otherwise it won't ask you the questions the next time you do a gradio deploy)

#### My Digital Twin

So I spend some time taking this project to the next level!  
Here it is:   
https://edwarddonner.com/avatar

Not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg

## The first big project - Professionally You!

### And, Tool use.

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the last lab, and add other Agentic patterns.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your career alter-ego) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>